In [77]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('dataset.csv')
df.head(25)

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1696 entries, 0 to 1695
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   variant_id      1696 non-null   int64 
 1   variant_nama    1696 non-null   object
 2   satuan_display  1696 non-null   object
 3   tanggal         1696 non-null   object
 4   harga           1696 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 66.4+ KB


In [80]:
df['harga'].isna().sum()

np.int64(0)

In [81]:

df['tanggal'] = pd.to_datetime(df['tanggal'])
df = df.sort_values(by=['variant_id', 'tanggal']).reset_index(drop=True)
df['harga'] = df['harga'].replace(0, np.nan)

Linear Interpolation (If some data have a value 0, get a value for this data [day x] of data day x-1 and data day x+1)

In [82]:
df['harga'] = df.groupby('variant_id')['harga'].transform(
    lambda x: x.interpolate(method='linear').ffill().bfill()
)

In [83]:
df.head(5)

,variant_id,variant_nama,satuan_display,tanggal,harga
0,2,Cabai Merah Keriting,kg,2026-01-01,47274.0
1,2,Cabai Merah Keriting,kg,2026-01-02,47274.0
2,2,Cabai Merah Keriting,kg,2026-01-03,46609.0
3,2,Cabai Merah Keriting,kg,2026-01-04,45944.0
4,2,Cabai Merah Keriting,kg,2026-01-05,45279.0


Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sample_commodities = ['Beras Medium', 'Daging Ayam Ras', 'Telur Ayam Ras', 'Bawang Merah']

for commodity in sample_commodities:
    plt.figure(figsize=(14, 6))
    plot_data = df[df['variant_nama'] == commodity]
    plt.plot(plot_data['tanggal'], plot_data['harga'], marker='o', markersize=3)

    plt.title(f'{commodity} Price Trends', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Price (IDR)', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()


Featured Engineering

In [85]:
df['day'] = df['tanggal'].dt.day
df['day_of_week'] = df['tanggal'].dt.dayofweek # 0=Monday, 6=Sunday
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Lag Features
# Grouping by 'variant_id' prevents data leakage between different commodities
df['price_lag_1'] = df.groupby('variant_id')['harga'].shift(1)
df['price_lag_2'] = df.groupby('variant_id')['harga'].shift(2)
df['price_lag_3'] = df.groupby('variant_id')['harga'].shift(3)

# Rolling Statistics
# Calculate 7-day moving average to capture short-term trends
df['rolling_mean_7d'] = df.groupby('variant_id')['harga'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

In [86]:
df_fe = df.dropna().reset_index(drop=True)